# OSINT Self-Play Training on Colab

This notebook clones the repo, installs the train dependencies, reads `HF_TOKEN` and `WANDB_API_KEY` from Colab Secrets, writes Colab-specific config files, and runs the same self-play runner used by `osint-env train-self-play`.

Before running:
- Switch the Colab runtime to GPU.
- Add `HF_TOKEN` in Colab Secrets.
- Add `WANDB_API_KEY` if you want W&B logging.
- Add `OPENAI_API_KEY` only if you change `ENV_LLM_PROVIDER` to `"openai"`.

The default notebook path is the lighter Colab setup: fixed canonical graphs, shared model topology, and LoRA-friendly phase overrides.

In [ ]:
REPO_URL = "https://github.com/RitishShrirao/OSINT_env.git"
REPO_REF = "main"
REPO_DIR = "/content/meta-knowledge-graph"

BASE_ENV_CONFIG = "config/shared_config.json"
BASE_TRAIN_CONFIG = "config/self_play_training_hf_a10g_smoke.json"

OUTPUT_DIR = "/content/artifacts/self_play_colab"
ROUNDS_OVERRIDE = 1
FORCE_DRY_RUN = False

ENV_LLM_PROVIDER = "mock"  # use "openai" only if OPENAI_API_KEY is set in Colab Secrets
ENV_LLM_MODEL = "gpt-4o-mini"
HF_CHECKPOINT_REPO_ID = ""  # optional: "username/osint-self-play-colab"
WANDB_PROJECT = "osint-self-play-train"
WANDB_ENTITY = ""
RUN_NAME_PREFIX = "colab-self-play"

USE_COLAB_SAFE_OVERRIDES = True
SAFE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SAFE_TUNING_MODE = "lora"
SAFE_NUM_GENERATIONS = 2
SAFE_BATCH_SIZE = 1
SAFE_GRAD_ACCUM = 4
SAFE_GENERATION_BATCH_SIZE = 4
SAFE_MAX_STEPS = 20
SAFE_SAVE_STEPS = 20

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/osint-self-play"


In [ ]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_REF, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_REF], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], cwd=REPO_DIR, check=True)


In [ ]:
import os

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("This notebook expects to run inside Google Colab.") from exc

def load_secret(name: str, required: bool = False) -> str:
    try:
        value = userdata.get(name)
    except Exception:
        value = ""
    token = str(value or "").strip()
    if token:
        os.environ[name] = token
    if required and not token:
        raise ValueError(f"Missing required Colab secret: {name}")
    return token

hf_token = load_secret("HF_TOKEN", required=True)
wandb_api_key = load_secret("WANDB_API_KEY", required=False)
openai_api_key = load_secret("OPENAI_API_KEY", required=False)

os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "/content/.cache/huggingface/transformers")

if HF_CHECKPOINT_REPO_ID:
    os.environ["OSINT_HF_CHECKPOINT_REPO_ID"] = HF_CHECKPOINT_REPO_ID
if WANDB_PROJECT:
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)

if wandb_api_key:
    import wandb
    wandb.login(key=wandb_api_key)
    print("W&B login configured.")
else:
    print("WANDB_API_KEY not found; the generated training config will disable W&B reporting.")


In [ ]:
import json
from pathlib import Path

import torch

repo_dir = Path(REPO_DIR)
env_config_path = repo_dir / BASE_ENV_CONFIG
train_config_path = repo_dir / BASE_TRAIN_CONFIG
generated_dir = repo_dir / "config" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)

with env_config_path.open("r", encoding="utf-8") as handle:
    env_payload = json.load(handle)
with train_config_path.open("r", encoding="utf-8") as handle:
    train_payload = json.load(handle)

env_payload.setdefault("llm", {})
env_payload["llm"]["provider"] = ENV_LLM_PROVIDER
if ENV_LLM_PROVIDER == "openai":
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY is required when ENV_LLM_PROVIDER is set to 'openai'.")
    env_payload["llm"]["model"] = ENV_LLM_MODEL
    env_payload["llm"]["openai_api_key"] = ""
    env_payload["llm"]["openai_api_key_env"] = "OPENAI_API_KEY"
else:
    env_payload["llm"]["provider"] = "mock"

supports_bf16 = bool(
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

output_root = DRIVE_OUTPUT_DIR if MOUNT_DRIVE else OUTPUT_DIR
Path(output_root).mkdir(parents=True, exist_ok=True)

train_payload["output_dir"] = output_root
train_payload["rounds"] = int(ROUNDS_OVERRIDE)
train_payload["dry_run"] = bool(FORCE_DRY_RUN)
train_payload["wandb_enabled"] = bool(wandb_api_key)
train_payload["wandb_project"] = WANDB_PROJECT or train_payload.get("wandb_project", "osint-self-play-train")
train_payload["wandb_entity"] = WANDB_ENTITY
train_payload["wandb_run_name_prefix"] = RUN_NAME_PREFIX
train_payload["pipeline_mode"] = "swarm_v2"
train_payload["canonical_graph_mode"] = "fixed"
train_payload["model_topology"] = "shared"
train_payload["shared_model_name_or_path"] = SAFE_MODEL_NAME

if USE_COLAB_SAFE_OVERRIDES:
    train_payload["tuning_mode"] = SAFE_TUNING_MODE
    train_payload["shared_model_name_or_path"] = SAFE_MODEL_NAME
    for phase_name in ("generator_phase", "answerer_phase"):
        phase = train_payload.setdefault(phase_name, {})
        phase["model_name_or_path"] = SAFE_MODEL_NAME
        phase["max_steps"] = SAFE_MAX_STEPS
        phase["per_device_train_batch_size"] = SAFE_BATCH_SIZE
        phase["gradient_accumulation_steps"] = SAFE_GRAD_ACCUM
        phase["num_generations"] = SAFE_NUM_GENERATIONS
        phase["generation_batch_size"] = SAFE_GENERATION_BATCH_SIZE
        phase["save_steps"] = SAFE_SAVE_STEPS
        phase["dataloader_num_workers"] = 0
        phase["dataloader_persistent_workers"] = False
        phase["bf16"] = supports_bf16
        phase["tf32"] = bool(torch.cuda.is_available())

generated_env_config = generated_dir / "colab_shared_config.json"
generated_train_config = generated_dir / "colab_self_play_config.json"
generated_env_config.write_text(json.dumps(env_payload, indent=2, sort_keys=True), encoding="utf-8")
generated_train_config.write_text(json.dumps(train_payload, indent=2, sort_keys=True), encoding="utf-8")

print(
    json.dumps(
        {
            "generated_env_config": str(generated_env_config),
            "generated_train_config": str(generated_train_config),
            "output_dir": output_root,
            "wandb_enabled": bool(wandb_api_key),
            "env_llm_provider": env_payload["llm"]["provider"],
            "supports_bf16": supports_bf16,
        },
        indent=2,
        sort_keys=True,
    )
)


In [ ]:
import platform

import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
supports_bf16 = bool(
    torch.cuda.is_available()
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

print(f"Python: {platform.python_version()}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {gpu_name}")
print(f"BF16 supported: {supports_bf16}")

if not torch.cuda.is_available() and not FORCE_DRY_RUN:
    print("Warning: no GPU runtime detected. Consider enabling a GPU or setting FORCE_DRY_RUN = True.")
elif torch.cuda.is_available() and not supports_bf16 and not FORCE_DRY_RUN:
    print("Warning: this runtime does not report bf16 support. The current trainer config may run slowly or need dry-run mode on this GPU.")


In [ ]:
import json
import os
import sys
from pathlib import Path

os.chdir(REPO_DIR)
src_dir = str(Path(REPO_DIR) / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from osint_env.config import clone_environment_config, load_shared_config
from osint_env.training import load_self_play_config, run_adversarial_self_play

shared_cfg = load_shared_config(str(generated_env_config))
env_cfg = clone_environment_config(shared_cfg.environment)
train_cfg = load_self_play_config(str(generated_train_config))

payload = run_adversarial_self_play(
    env_config=env_cfg,
    training_config=train_cfg,
    dry_run=bool(FORCE_DRY_RUN),
)

print(
    json.dumps(
        {
            "summary_path": payload.get("summary_path"),
            "final_models": payload.get("final_models", {}),
            "post_training_evaluation": payload.get("post_training_evaluation", {}),
        },
        indent=2,
        sort_keys=True,
    )
)


In [ ]:
import json
from pathlib import Path

summary = json.loads(Path(payload["summary_path"]).read_text(encoding="utf-8"))
round_rows = []
for round_payload in summary.get("rounds", []):
    round_rows.append(
        {
            "round": round_payload.get("round"),
            "generated_task_count": round_payload.get("generated_task_count"),
            "answerer_task_count": round_payload.get("answerer_task_count"),
            "generator_model": (round_payload.get("generator") or {}).get("model_path"),
            "answerer_model": (round_payload.get("answerer") or {}).get("model_path"),
        }
    )

print(json.dumps({"rounds": round_rows}, indent=2, sort_keys=True))
print(json.dumps(summary.get("post_training_evaluation", {}), indent=2, sort_keys=True))
